## 0. Install dependencies

This notebook requires ``iqm-qubit-selector`` and ``iqm-benchmarks``. Run the cell below once per environment; re-running it is
harmless if the package is already installed.

In [ ]:
%pip install --quiet iqm-qubit-selector iqm-benchmarks matplotlib

# Levels of error reduction on a GHZ state

[3_ghz_benchmark.ipynb](3_ghz_benchmark.ipynb) measured a GHZ state prepared the lazy way: on the first
``n`` qubits, with no mitigation. This notebook starts one step further along - with the **tree circuit**,
which is the sensible default for preparing a GHZ state on real hardware - and then adds one error
reduction technique at a time, each level keeping everything the previous ones turned on.

Every level is the same benchmark, on the same device, with one thing changed. That is the whole point: the
difference in fidelity between two consecutive levels is attributable to exactly one technique. Each level
writes its ``GHZConfiguration`` out in full rather than copying the previous one, so what changed is
visible in the cell you are looking at.

| Level | What it changes | Field |
|---|---|---|
| starting point | build the state along the best connections | ``state_generation_routine="tree"`` |
| 1 | protect idling qubits, three strategies compared | ``use_dd`` and ``dd_strategy`` |
| 2 | move to well calibrated qubits | ``custom_qubits_array`` |
| 3 | correct the measurement | ``rem`` |

1. **Setup** - imports, token, backend, and the one helper carried over from notebook 3.
2. **The starting point** - the tree circuit, and the fidelity everything else is measured against.
3. **Level 1: dynamical decoupling** - three strategies measured against each other, best one carried on.
4. **Level 2: qubit selector** - move the whole thing onto well calibrated qubits.
5. **Level 3: readout error mitigation** - correct the measurement itself.
6. **The ladder** - all levels side by side.

## 1. Setup

Three IQM packages in the same environment: ``iqm-benchmarks`` for the benchmark, ``iqm-qubit-selector``
for the layout choice in level 2, and ``iqm-client[qiskit]`` for the backend.

In [ ]:
import os

from qiskit import QuantumCircuit

from iqm.benchmarks.entanglement.ghz import GHZBenchmark, GHZConfiguration
from iqm.iqm_client.models import DDStrategy
from iqm.qiskit_iqm import IQMProvider
from iqm.qubit_selector.qubit_selector import CalibrationDataManager, CalibrationType, CostEvaluator

## Plotting helpers for this demo, defined next to this notebook.
from utils import apply_demo_style, plot_fidelity_levels
from utils_benchmark import ghz_tree_edges, plot_cz_graph, plot_tree_comparison

apply_demo_style()  ## also restyles the plots that iqm-benchmarks produces itself

The token is read from the ``IQM_TOKEN`` environment variable. Export it in your shell before starting
Jupyter; setting it inside the notebook is convenient but makes it easy to expose the token accidentally,
for example when presenting or when committing the notebook to a shared repository.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()  # loads IQM_TOKEN from a local .env file into os.environ
token = os.getenv("IQM_TOKEN")
quantum_computer = "emerald" 
iqm_server_url = "https://resonance.iqm.tech/"  # provide your actual IQM server URL
os.environ["IQM_SERVER_URL"] = iqm_server_url
provider = IQMProvider(iqm_server_url, quantum_computer=quantum_computer)
backend = provider.get_backend()

print(f"Connected to {quantum_computer} with {backend.num_qubits} qubits.")

### One helper, and the parameters shared by every level

The only thing worth hiding in a function is ``ghz_fidelity``, which filters one layout's fidelity out of
an analysis result - it is the same helper as in notebook 3. Everything else is written out at each level,
so you can see the whole configure-run-analyse cycle every time:

```python
benchmark = GHZBenchmark(backend, CONFIGURATION)
benchmark.run()        ## builds, transpiles and submits the circuits
result = benchmark.analyze()   ## counts -> observations and plots
```

In [ ]:
nqubits = 30
shots = 1000

if nqubits > backend.num_qubits:
    raise ValueError(f"{quantum_computer} has {backend.num_qubits} qubits, cannot host a {nqubits}-qubit GHZ state")

naive_layout = list(range(backend.num_qubits - nqubits, backend.num_qubits))  ## the last n qubits


def ghz_fidelity(result, qubit_layout, name="fidelity"):
    """Pick one fidelity observation of one layout out of an analysis result."""
    for observation in result.observations:
        if observation.identifier.string_identifier == str(list(qubit_layout)) and observation.name == name:
            return observation.value
    raise KeyError(f"No {name} observation for layout {qubit_layout}")


levels = {}  ## the cumulative ladder: one entry per error reduction step
dd_variants = {}  ## the decoupling strategies of level 1, compared against each other

## 2. The starting point: the tree circuit

The first thing to fix about a GHZ state is not *where* it is prepared but *how*.
``state_generation_routine="tree"`` is what this notebook uses throughout, and it is the starting point
rather than one of the levels.

The tree routine builds a minimum spanning tree over the CZ fidelities inside the layout and roots it to
minimise the largest weighted distance, so the entanglement spreads along the **best gates available** and
the depth stays logarithmic in $n$ rather than linear. Shorter depth means less time for the state to
decohere, and routing along good CZs means fewer errors injected on the way. Because it reads the current
calibration, the circuit it produces can differ between runs as the device is recalibrated.

The graph below shows the tree it will build on the naive layout: edge width is the CZ error, so a **thin**
highlighted edge is a good gate the tree managed to find.

In [ ]:
tree_on_naive = ghz_tree_edges(backend, naive_layout)

ax = plot_cz_graph(
    backend,
    naive_layout,
    tree_on_naive,
    title=f"The GHZ tree on the naive {nqubits}-qubit layout",
)

In [ ]:
TREE = GHZConfiguration(
    state_generation_routine="tree",  ## build the state along the best CZ connections
    custom_qubits_array=[naive_layout],
    shots=shots,
    fidelity_routine="coherences",
    rem=False,  ## no readout error mitigation yet
    use_dd=False,  ## no dynamical decoupling yet
    max_circuits_per_batch=100,
)

benchmark_tree = GHZBenchmark(backend, TREE)
benchmark_tree.run()  ## builds, transpiles and submits the circuits
result_tree = benchmark_tree.analyze()  ## counts -> observations and plots

levels["Tree circuit"] = ghz_fidelity(result_tree, naive_layout)

print(f"GHZ fidelity with the tree circuit: {levels['Tree circuit']:.4f}")

## 3. Level 1: dynamical decoupling

In a GHZ circuit most qubits spend most of their time idling, waiting for the entanglement to reach them,
and an idling qubit dephases. Dynamical decoupling fills those gaps with pulses that echo the dephasing
away.

It is not free, and not always a win. Whether it helps depends on

- how many idle segments the circuit has, and how long they are,
- how large the idling errors are compared to gate errors,
- the single-qubit gate fidelity, since DD *adds* single-qubit gates.

Because none of that is knowable in advance, this level does not pick a strategy - it **measures three**,
all on the tree circuit and the same qubits, changing nothing but the decoupling:

| Variant | Qubits | Sequence |
|---|---|---|
| **(a) standard** | every qubit | whatever the compiler's default strategy chooses per idle window |
| **(b) targeted XX** | only well calibrated qubits | two X gates at the centre of the window |
| **(c) targeted XY4** | only well calibrated qubits | ``YXYX``, echoing both axes |

A ``DDStrategy`` is a list of ``(ratio, gate pattern, align)`` sequences plus an optional
``target_qubits`` set. The ``ratio`` is the minimum window length the sequence needs, measured in PRX gate
durations, and the compiler picks the longest sequence that fits each idle window. So **XX** (ratio 2) fits
into short gaps, while **XY4** (ratio 5) needs a window more than twice as long but cancels noise along
both axes instead of one.

Comparing (a) with (b) isolates *where* the pulses go; comparing (b) with (c) isolates *which* sequence
runs. The winner is what the rest of the ladder builds on.

In [ ]:
calibration_data = CalibrationDataManager().get_calibration_fidelities(backend)

## Qubits whose single-qubit gates are good enough for the added DD pulses to pay for themselves.
## The keys are already physical qubit names, which is exactly what DDStrategy expects.
good_sq_qubits = frozenset(
    qubit for qubit, sq_fidelity in calibration_data[CalibrationType.SQG.value].items() if sq_fidelity > 0.999
)


In [ ]:
## (b) two X gates at the centre of each idle window, only on the well calibrated qubits.
DD_XX = DDStrategy(gate_sequences=[(2, "XX", "center")], target_qubits=good_sq_qubits)

## (c) same qubits, but XY4 - needs a longer window and echoes both axes.
DD_XY4 = DDStrategy(gate_sequences=[(5, "YXYX", "asap")], target_qubits=good_sq_qubits)

## (a) needs no object at all: dd_strategy=None means "use the compiler's standard strategy".

print(f"Well calibrated qubits ({len(good_sq_qubits)} of {backend.num_qubits}): {sorted(good_sq_qubits)}")
print(f"(b) XX  : {DD_XX.gate_sequences}")
print(f"(c) XY4 : {DD_XY4.gate_sequences}")

In [ ]:
## (a) Out of the box: decoupling on, no strategy given, so every qubit gets the compiler's default.
DD_STANDARD = GHZConfiguration(
    state_generation_routine="tree",
    custom_qubits_array=[naive_layout],
    shots=shots,
    fidelity_routine="coherences",
    rem=False,
    use_dd=True,
    dd_strategy=None,  ## the compiler's standard strategy, on every qubit
    max_circuits_per_batch=100,
)

benchmark_dd_standard = GHZBenchmark(backend, DD_STANDARD)
benchmark_dd_standard.run()
result_dd_standard = benchmark_dd_standard.analyze()

dd_variants["(a) standard, all qubits"] = ghz_fidelity(result_dd_standard, naive_layout)
print(f"(a) standard, all qubits: {dd_variants['(a) standard, all qubits']:.4f}")

In [ ]:
## (b) Same circuit, same qubits measured - but the pulses only go on the well calibrated qubits.
DD_TARGETED_XX = GHZConfiguration(
    state_generation_routine="tree",
    custom_qubits_array=[naive_layout],
    shots=shots,
    fidelity_routine="coherences",
    rem=False,
    use_dd=True,
    dd_strategy=DD_XX,  ## XX at the centre of each window, well calibrated qubits only
    max_circuits_per_batch=100,
)

benchmark_dd_xx = GHZBenchmark(backend, DD_TARGETED_XX)
benchmark_dd_xx.run()
result_dd_xx = benchmark_dd_xx.analyze()

dd_variants["(b) targeted XX"] = ghz_fidelity(result_dd_xx, naive_layout)
print(f"(b) targeted XX: {dd_variants['(b) targeted XX']:.4f}")

In [ ]:
## (c) Only the sequence changes from (b): XY4 instead of XX, on the same qubits.
DD_TARGETED_XY4 = GHZConfiguration(
    state_generation_routine="tree",
    custom_qubits_array=[naive_layout],
    shots=shots,
    fidelity_routine="coherences",
    rem=False,
    use_dd=True,
    dd_strategy=DD_XY4,  ## YXYX, needs a longer idle window than XX
    max_circuits_per_batch=100,
)

benchmark_dd_xy4 = GHZBenchmark(backend, DD_TARGETED_XY4)
benchmark_dd_xy4.run()
result_dd_xy4 = benchmark_dd_xy4.analyze()

dd_variants["(c) targeted XY4"] = ghz_fidelity(result_dd_xy4, naive_layout)
print(f"(c) targeted XY4: {dd_variants['(c) targeted XY4']:.4f}")

### Which decoupling actually helped?

Four numbers on the same circuit and the same qubits: no decoupling, and the three strategies. Any of them
can come out behind the undecoupled run - that is the honest outcome when the added single-qubit gates cost
more than the idling they remove, and it is exactly why this is measured rather than assumed.

The best of the three is what level 2 and level 3 build on.

In [ ]:
print(f"{'no decoupling':<26}{levels['Tree circuit']:.4f}")
for name, value in dd_variants.items():
    print(f"{name:<26}{value:.4f}  ({value - levels['Tree circuit']:+.4f})")

## Carry the best strategy forward. The DDStrategy object that produced it goes into the next levels.
best_dd_name = max(dd_variants, key=dd_variants.get)
best_dd_strategy = {
    "(a) standard, all qubits": None,
    "(b) targeted XX": DD_XX,
    "(c) targeted XY4": DD_XY4,
}[best_dd_name]

levels["+ dynamical decoupling"] = dd_variants[best_dd_name]
print(f"\nBest: {best_dd_name} -> carried into levels 2 and 3")

## 4. Level 2: choose the qubits with the qubit selector

Everything so far ran on ``list(range(n))``, which is not a considered choice - it is just the first ``n``
qubits. ``iqm-qubit-selector`` ranks the layouts that can host a circuit against the current calibration
data, so we hand it the GHZ circuit and take the best layout it returns. Costs come back as errors, so
lower is better, and a layout is a list of qiskit indices - exactly what ``custom_qubits_array`` expects.

Two things worth saying out loud at this point:

- The selector is asked about the *textbook chain*, because that fixes the interaction pattern it has to
  fit onto the chip. The tree routine then rebuilds the state inside whichever qubits it picked, so the two
  cooperate rather than compete.
- This is the one level that changes which physical qubits are measured, so from here on the fidelity is
  read off ``selected_layout`` rather than ``naive_layout``.

In [ ]:
## The circuit we are asking about: the textbook GHZ chain on nqubits.
ghz_circuit = QuantumCircuit(nqubits)
ghz_circuit.h(0)
for i in range(1, nqubits):
    ghz_circuit.cx(i - 1, i)
ghz_circuit.measure_all()

selector_layouts, selector_costs = CostEvaluator(backend, ghz_circuit, num_trials=5000).get_top_layouts(num_layouts=10)
selected_layout = selector_layouts[0]

print(f"Naive layout:    {naive_layout}")
print(f"Selected layout: {selected_layout} -> {[backend.index_to_qubit_name(q) for q in selected_layout]}")
#print(f"Predicted cost:  {selector_costs[0] * 100:.2f}%")

In [ ]:
## The same chip twice, with the CZ gates each tree would use drawn on top.
## Edge width is the CZ error, so a thin highlighted edge is a good gate the tree managed to find.
ax_naive, ax_selected = plot_tree_comparison(backend, naive_layout, selected_layout)

In [ ]:
## Everything is as in level 1 except custom_qubits_array, which now holds the selected layout.
SELECTOR = GHZConfiguration(
    state_generation_routine="tree",
    custom_qubits_array=[selected_layout],  ## the one field that changes
    shots=shots,
    fidelity_routine="coherences",
    rem=False,
    use_dd=True,
    dd_strategy=best_dd_strategy,  ## the winner of level 1
    max_circuits_per_batch=100,
)

benchmark_selector = GHZBenchmark(backend, SELECTOR)
benchmark_selector.run()
result_selector = benchmark_selector.analyze()

levels["+ qubit selector"] = ghz_fidelity(result_selector, selected_layout)

print(f"On the naive qubits:    {levels['+ dynamical decoupling']:.4f}")
print(f"On the selected qubits: {levels['+ qubit selector']:.4f}")

## 5. Level 3: readout error mitigation

Everything so far protected the state. Readout error mitigation instead corrects the *measurement*: it
spends ``mit_shots`` characterising how often each qubit is misread, then inverts that on the counts.

This one is nearly free to evaluate, because with ``rem=True`` the analysis reports **both** numbers from
the same run - ``fidelity`` as measured and ``fidelity_rem`` after mitigation - so you can see exactly what
the correction is worth on your data.

In [ ]:
## Everything is as in level 2 except the two readout mitigation fields.
REM = GHZConfiguration(
    state_generation_routine="tree",
    custom_qubits_array=[selected_layout],
    shots=shots,
    fidelity_routine="coherences",
    rem=True,  ## correct the measurement
    mit_shots=1000,  ## shots spent characterising the readout
    use_dd=True,
    dd_strategy=best_dd_strategy,
    max_circuits_per_batch=100,
)

benchmark_rem = GHZBenchmark(backend, REM)
benchmark_rem.run()
result_rem = benchmark_rem.analyze()

raw = ghz_fidelity(result_rem, selected_layout, name="fidelity")
mitigated = ghz_fidelity(result_rem, selected_layout, name="fidelity_rem")
levels["+ readout mitigation"] = mitigated

print(f"Same run, before mitigation: {raw:.4f}")
print(f"Same run, after mitigation:  {mitigated:.4f}")

## 6. The ladder

Three techniques on top of the tree circuit, each added to the previous one, all measured with the same
benchmark on the same device. The leftmost bar is the starting point rather than a lazy baseline, so every
step to the right of it is a technique paying for itself - or not. The dashed line at 0.5 is the
entanglement threshold: above it, the state is certified as genuinely multipartite entangled.

In [ ]:
for name, value in levels.items():
    print(f"{name:<24}{value:.4f}")

ax = plot_fidelity_levels(
    levels,
    title=f"{nqubits}-qubit GHZ fidelity on {quantum_computer}",
    subtitle=f"{shots} shots per circuit, coherences routine. Each step keeps the previous ones",
)

## Recap

- The **tree circuit** is where to start, not a technique to add: it is logarithmic in depth and routes the
  entanglement along the best CZ connections the layout offers, at the cost of depending on today's
  calibration.
- The three techniques on top of it attack different error sources, which is why they compose:
  **dynamical decoupling** protects the qubits that wait while the state is built, **the qubit selector**
  moves the whole thing onto better qubits, and **readout mitigation** fixes the measurement at the end.
- Each level writes its configuration out in full, so the diff between two levels is visible in the cell
  itself - and the fidelity difference is attributable to exactly that diff.
- **Dynamical decoupling is a family, not a switch.** Where the pulses go and which sequence runs both
  matter: the standard strategy pulses every qubit, while a targeted strategy spends gates only where the
  single-qubit fidelity can pay for them, and XX and XY4 fit different idle windows. Level 1 measured all
  three rather than assuming one.
- None of them is unconditional. Decoupling adds gates that only pay off when idling dominates, and the
  qubit selector can only help if the chip is inhomogeneous today and the layout is smaller than the chip -
  which is why measuring each level, rather than assuming, is the point of this notebook.
- Bigger GHZ states are the real test: raise ``nqubits`` and re-run the ladder to find where the device
  stops crossing 0.5, with and without the mitigations.